# Urdu Orpheus on Kaggle — complete fresh-runtime setup
Import **this updated notebook** into Kaggle; updating the cloned repository does not update existing notebook cells.
Enable GPU and Internet, attach the ASLP dataset via **Add Input** (Kaggle Dataset), add the HF_TOKEN Secret, and edit HF_REPO_ID and SESSION_HOURS in Cell 1. The repository URL is already filled in.
Cell 2 always installs dependencies before checking imports. Do not replace it with an import-only cell.
Use Save Version → Save & Run All for unattended execution. Run a separate smoke experiment before the full epoch; see README.md.
Default policy skips small bad/duplicate/context-overflow subsets and logs every exclusion. Training retention limits are in configs/aslp50h.json.
Recovery now queues failed uploads locally, regenerates damaged encoding caches and probes GPU backward memory automatically. See REVIEW.md for tested coverage and remaining limits.

Hugging Face checkpoint backups upload files directly without a local tar copy. Disk checks reserve the next save; older verified checkpoints are pruned under pressure while the newest is preserved.

Checkpoint saving is exercised at optimizer step 1. Optional monitoring starts after a saved checkpoint by default. Final export reuses that checkpoint’s adapter weights.


Cell 1 — clone a fixed version

In [ ]:
import os, sys, subprocess, re
from pathlib import Path

REPO_URL = "https://github.com/hassan-31x/orpheus-ft-urdu.git"
REPO_REF = "43faf05cacf57ea98b700a709eb24aab04ac87ac"  # Adds INIT_FROM_RUN (epoch 2). Keep it fixed for this run and its resumes.
HF_REPO_ID = "hassan-31x/orpheus-urdu-checkpoints"  # Your private model repository.
RUN_ID_OVERRIDE = "aslp50h-r32-lr1e4-seed3407-v2-epoch2"  # Epoch-2 run; keep the same name to resume it.
# EPOCH 2: start from the FINISHED epoch-1 run (its adapter + token cache; fresh optimizer/schedule).
# Set to None for a from-scratch run (then use the epoch-1 name above as RUN_ID_OVERRIDE).
INIT_FROM_RUN = "aslp50h-r32-lr1e4-seed3407-v2"
LEARNING_RATE = 5e-5  # Lower than epoch 1 (1e-4) so the second pass refines instead of overwriting. None = config default.
AUDIT_ONLY = False  # Set True with GPU OFF to inspect data before training.
# TEST=True: quick end-to-end run on 50 training clips for one epoch (~7 optimizer steps,
# checkpoints at steps 1/5/7, validation, samples, Hub upload, final adapter). Separate run_id,
# so it never touches the full run. When it prints TEST PASSED, set TEST=False for the full epoch.
TEST = False
TEST_TRAIN_CLIPS = 50
MINIMUM_TRAIN_HOURS = 35  # The 47h archive contains ~38h train plus validation/test.
SESSION_HOURS = 10.5  # Set below the limit displayed in your Kaggle account.
# Dataset: attach the ASLP archive as a Kaggle Dataset (notebook editor -> Add Input).
# It is found automatically under /kaggle/input. Google Drive downloading is blocked
# when the shared file exceeds Drive's download quota, and every resume would re-download.
DATA_DIR = None  # None = auto-detect under /kaggle/input; or the folder containing train.csv
ALLOW_DRIVE_DOWNLOAD = False  # True only if no Kaggle Dataset is attached
REPO = Path("/kaggle/working/urdu-orpheus")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", HF_REPO_ID):
    raise ValueError("HF_REPO_ID must have the format username/repository")
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Add the HF_TOKEN Kaggle Secret and enable access for this notebook") from None
MANIFEST_NAMES = {"stage1_train.csv", "train.csv", "train_manifest.csv"}
if DATA_DIR is None and Path("/kaggle/input").is_dir():
    found = sorted({p.parent for p in Path("/kaggle/input").rglob("*.csv") if p.name in MANIFEST_NAMES})
    if len(found) > 1:
        raise RuntimeError(f"Several training manifests under /kaggle/input: {found}. Set DATA_DIR to one of them")
    DATA_DIR = found[0] if found else None
if DATA_DIR is not None:
    DATA_DIR = Path(DATA_DIR)
    if not any((DATA_DIR / name).is_file() for name in MANIFEST_NAMES):
        raise RuntimeError(f"DATA_DIR={DATA_DIR} has no {sorted(MANIFEST_NAMES)}")
    print("Dataset (Kaggle Input):", DATA_DIR, "-", sum(1 for _ in DATA_DIR.rglob("*.wav")), "WAV files")
elif ALLOW_DRIVE_DOWNLOAD:
    print("No Kaggle Dataset attached; will download from Google Drive (may hit Drive's quota)")
else:
    zips = sorted(str(p) for p in Path("/kaggle/input").rglob("*.zip")) if Path("/kaggle/input").is_dir() else []
    raise RuntimeError("No dataset attached. Notebook editor -> Add Input -> Your Datasets -> the ASLP dataset, "
                       "then Save Version again." + (f" Found unextracted ZIPs {zips}: re-upload so Kaggle extracts "
                       "them, or set ALLOW_DRIVE_DOWNLOAD=True." if zips else ""))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if not AUDIT_ONLY:
    # Fail in seconds, before installation, when the session has no GPU attached.
    try:
        gpu_list = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    except FileNotFoundError:
        gpu_list = ""
    if "GPU" not in gpu_list:
        raise RuntimeError("No GPU in this session. Notebook editor -> Settings -> Accelerator -> GPU, then "
                           "Save Version again. Kaggle may not attach a GPU when your weekly GPU quota is used "
                           "up or your concurrent GPU session limit is reached.")
    print(gpu_list.strip())
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # progress bars flood logs
os.environ["TQDM_DISABLE"] = "1"

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
# Fetch even when the directory already exists; checkout alone leaves old code.
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
for name in ("setup_kaggle.py", "check_environment.py", "requirements-kaggle.txt", "pipeline_recovery.py"):
    if not (REPO / name).is_file():
        raise RuntimeError(f"Repository version lacks {name}. Upload the updated project before running.")
PROJECT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Project commit:", PROJECT_COMMIT)

# Fast Hub check BEFORE the long install: a token/permission problem fails here in seconds.
# Uses Kaggle's preinstalled huggingface_hub; Cell 3 repeats it inside the training venv.
if (REPO / "check_hub.py").is_file():
    hub_check = [sys.executable, str(REPO / "check_hub.py"), "--repo", HF_REPO_ID]
    if RUN_ID_OVERRIDE:
        hub_check += ["--run-id", RUN_ID_OVERRIDE]
    result = subprocess.run(hub_check, cwd=REPO)
    if result.returncode == 2:
        raise RuntimeError("Hugging Face storage check failed; see HUB CHECK FAILED diagnostics above")
    if result.returncode:
        print("Early Hub check could not run with system Python; Cell 3 will check inside the venv")


Cell 2 — install and preflight

In [ ]:
# Always install in this runtime: saved Kaggle jobs start with a fresh environment.
ENV = Path("/kaggle/working/orpheus-env")
TRAIN_PYTHON = str(ENV / "bin" / "python")
setup_command = [sys.executable, "-u", str(REPO / "setup_kaggle.py"), "--venv", str(ENV)]
if not AUDIT_ONLY:
    setup_command.append("--gpu")
subprocess.run(setup_command, cwd=REPO, check=True)
assert Path(TRAIN_PYTHON).is_file(), "Training environment was not created"
print("All training subprocesses will use:", TRAIN_PYTHON)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)


Cell 3 — private credentials and config

In [ ]:
import json
cfg = json.loads((REPO / "configs/aslp50h.json").read_text())
if RUN_ID_OVERRIDE:
    cfg["run_id"] = RUN_ID_OVERRIDE
if TEST:
    # One test run per code commit: a checkpoint from older code can never block a new test.
    cfg["run_id"] = cfg["run_id"] + f"-smoke-test{TEST_TRAIN_CLIPS}-{PROJECT_COMMIT[:7]}"
    cfg.update(test_train_clips=TEST_TRAIN_CLIPS, test_validation_clips=10,
               save_steps=5, eval_steps=5, logging_steps=1)
    print(f"TEST MODE: {TEST_TRAIN_CLIPS} training clips, 10 validation clips, run_id={cfg['run_id']}")
if DATA_DIR is not None:
    cfg["data_dir"] = str(DATA_DIR)  # read in place: no Drive download, no extraction copy
if INIT_FROM_RUN:
    cfg["init_from_run"] = INIT_FROM_RUN
    print("Continuing from finished run:", INIT_FROM_RUN)
if LEARNING_RATE is not None:
    cfg["learning_rate"] = LEARNING_RATE
cfg["checkpoint_backend"] = "huggingface"
cfg["hf_repo_id"] = HF_REPO_ID
cfg["session_hours"] = SESSION_HOURS
cfg["minimum_train_hours"] = MINIMUM_TRAIN_HOURS
print("Minimum retained TRAIN split hours:", cfg["minimum_train_hours"])
print("Retention thresholds:", "enforced" if cfg["enforce_retention_limits"] else "warnings only")
print("Data policy:", cfg["invalid_row_policy"], "maximum training exclusion fraction:", cfg["maximum_rejected_train_fraction"])
CONFIG = Path("/kaggle/working/run-config.json")
CONFIG.write_text(json.dumps(cfg, indent=2))
RUN = Path(cfg["work_dir"]) / "runs" / cfg["run_id"]
subprocess.run([TRAIN_PYTHON, str(REPO / "finetune_aslp_50h.py"),
                "--config", str(CONFIG), "--mode", "storage-check"], cwd=REPO, check=True)


Cell 4A — recommended unattended saved run

In [ ]:
# Blocks until the audit/training process completes. Failures print diagnostics.
command = [TRAIN_PYTHON, "-u", str(REPO / "finetune_aslp_50h.py"), "--config", str(CONFIG)]
if AUDIT_ONLY:
    command += ["--mode", "audit"]
import time

CONSOLE = Path("/kaggle/working/train_console.log")

def show_tail(lines):
    try:
        with CONSOLE.open("rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 200_000))
            print("\n".join(f.read().decode("utf-8", "replace").splitlines()[-lines:]), flush=True)
    except OSError:
        pass

# Training output goes to a FILE, never the notebook stream: a stalled Kaggle log
# pipeline can block a process that writes to it. Progress tail printed every 5 min;
# full log: /kaggle/working/train_console.log (and monitoring/ on Hugging Face).
print("Training output ->", CONSOLE, flush=True)
try:
    with CONSOLE.open("ab") as console:
        process = subprocess.Popen(command, cwd=REPO, stdout=console, stderr=subprocess.STDOUT,
                                   stdin=subprocess.DEVNULL)
        last_tail = time.time()
        while process.poll() is None:
            time.sleep(15)
            if time.time() - last_tail >= 300:
                print(f"--- {time.strftime('%H:%M:%S')} last lines ---", flush=True)
                show_tail(6)
                last_tail = time.time()
    show_tail(60)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)
except subprocess.CalledProcessError:
    import shutil
    print("Free disk GiB:", round(shutil.disk_usage(RUN).free / 2**30, 2))
    path = RUN / "disk_budget.json"
    if path.exists():
        print("Disk budget:", path.read_text())
    print("Training process exited. The traceback above is the failure; audit exclusions are separate.")
    if TEST:
        print("TEST FAILED. Keep TEST=True and fix the error above before a full run.")
    raise
if TEST and not AUDIT_ONLY:
    final_status = json.loads((RUN / "status.json").read_text()) if (RUN / "status.json").exists() else {}
    backup_info = json.loads((RUN / "backup_status.json").read_text()) if (RUN / "backup_status.json").exists() else {}
    wavs = sorted((RUN / "samples").glob("step-*/*.wav"))
    print("Status:", final_status.get("status"), "step:", final_status.get("step"))
    print("Validation:", final_status.get("validation"))
    print("Samples generated:", len(wavs), "| adapter_final exists:", (RUN / "adapter_final").is_dir())
    print("Remote backup:", backup_info)
    if (final_status.get("status") == "smoke_complete" and (RUN / "adapter_final").is_dir()
            and backup_info.get("remote_snapshot_current")):
        print("TEST PASSED: training, checkpoints, validation, Hub backup and final adapter all worked.")
        print("Set TEST = False in Cell 1 and Save & Run All for the full epoch.")
    else:
        print("TEST INCOMPLETE: see the values above (e.g. remote backup pending) before a full run.")


Cell 5 — status and latest samples

In [ ]:
from IPython.display import Audio, display

def optional_json(path):
    try:
        return json.loads(path.read_text()) if path.exists() else {}
    except (OSError, ValueError) as exc:
        print("Optional report unavailable:", path.name, type(exc).__name__)
        return {}

status_value = optional_json(RUN / "status.json")
print(status_value or ("Dataset audit only" if AUDIT_ONLY else "No training status recorded"))
for path in sorted((RUN / "samples").glob("step-*/*.wav"))[-3:]:
    print(path)
    try:
        display(Audio(filename=str(path)))
    except (OSError, ValueError) as exc:
        print("Optional playback unavailable:", type(exc).__name__)
backup = optional_json(RUN / "backup_status.json")
if backup:
    print("Remote backup:", backup)


Cell 6 — test the final adapter after completion